# Torsion Scan Guardian — Vast.ai single-molecule demo

Runs the Phase-5 active-learning demo on sulfanilamide on a Vast.ai GPU instance, driven from VSCode via Remote-SSH.

**Prerequisite:** you've followed [`VastAI_WAY.md`](../../VastAI_WAY.md) steps 1–7 to:
- Rent a Vast instance (recommend RTX 3090 on-demand, verified, reliability > 0.99)
- Connect VSCode to it via Remote-SSH (with the host alias `vast-torsion` in your local `~/.ssh/config`)
- Clone this repo on the instance
- Install Python deps (`pip install -e .` + xtb binary)
- Pre-warm the MACE-OFF cache

Wall time on RTX 3090: **~3–4 min**. Cost: ~$0.02 at $0.25/h.

## 1. Verify the environment

In [ ]:
import os, torch
print('cwd:', os.getcwd())
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert os.path.exists('pyproject.toml'), 'Run this notebook from the repo root (cd torsion_scan_guardian first).'
# Vast instances run xtb from a hand-installed location; confirm it's on PATH.
import shutil
assert shutil.which('xtb'), 'xtb not on PATH — see VastAI_WAY.md step 7 (Option A binary install)'
print('xtb:', shutil.which('xtb'))

In [ ]:
# Pre-warm MACE-OFF cache if not already (idempotent).
from mace.calculators import mace_off
mace_off(model='small', device='cuda' if torch.cuda.is_available() else 'cpu')
p = os.path.expanduser('~/.cache/mace/MACE-OFF23_small.model')
assert os.path.exists(p)
print('MACE-OFF cache:', os.path.getsize(p) // 1024, 'KB')

In [ ]:
!python -m pytest -q tests/ -k 'not test_ensemble_predict_h2o'

## 2. Phase 5 demo on sulfanilamide

Same config as REPORT §12.5. Wall time ~3–4 min on RTX 3090.

Cells 2a and 2b are idempotent — they skip if the artifacts already exist (useful if you're re-running after re-renting the instance).

In [ ]:
%%bash
# Step 2a: build the seed dataset (~2 min on RTX 3090).
if [ ! -f data/seed/sulfanilamide_seed.xyz ]; then
    python scripts/build_seed_dataset.py \
        --smiles 'Nc1ccc(S(=O)(=O)N)cc1' \
        --out data/seed/sulfanilamide_seed.xyz
else
    echo "seed dataset exists — skipping build"
fi

In [ ]:
%%bash
# Step 2b: fine-tune 3 ensemble members on GPU (~1 min total on RTX 3090).
for SEED in 0 1 2; do
    CKPT=runs/finetune_sulf/member_seed${SEED}/checkpoints/member_seed${SEED}_run-${SEED}.model
    if [ ! -f "$CKPT" ]; then
        MPLBACKEND=Agg python scripts/finetune_member.py \
            --seed $SEED --epochs 5 --lr 5e-4 \
            --train-file data/seed/sulfanilamide_seed.xyz \
            --out-root runs/finetune_sulf \
            --device cuda
    else
        echo "member $SEED already trained — skipping"
    fi
done
ls runs/finetune_sulf/*/checkpoints/*.model

In [ ]:
%%bash
# Step 2c: run the 5-cycle AL loop with cloud acquisitions + safeguarded fine-tune.
MPLBACKEND=Agg python -m guardian.cli \
    --config config/default.yaml \
    --smiles 'Nc1ccc(S(=O)(=O)N)cc1' \
    --steps 4000 --temperature 300 --threshold 0.05 \
    --max-triggers 5 --cooldown-steps 200 \
    --cloud-size 5 --cloud-jitter 0.05 \
    --ft-regression-tol 0.10 \
    --checkpoints runs/finetune_sulf/member_seed0/checkpoints/*.model \
                  runs/finetune_sulf/member_seed1/checkpoints/*.model \
                  runs/finetune_sulf/member_seed2/checkpoints/*.model \
    --online-finetune \
    --seed-data-file data/seed/sulfanilamide_seed.xyz \
    --finetune-epochs 2 --finetune-lr 1e-4 \
    --run-dir runs/sulf_phase5_vastai

In [ ]:
# Step 2d: analysis figures.
!MPLBACKEND=Agg python scripts/analyse_al_demo.py runs/sulf_phase5_vastai

In [ ]:
from IPython.display import Image
Image('figures/sulf_phase5_vastai_timeline.png')

## 3. Push results back to GitHub

Three options documented in [`VastAI_WAY.md`](../../VastAI_WAY.md) §10:
- **Option A — `scp` artifacts to local, push from local** (no creds on Vast). Cleanest.
- **Option B — fine-grained PAT on the Vast instance** (rotate / revoke when done).
- **Option C — SSH key on the Vast instance** (for repeated use of the same instance).

Example for Option A from your **local** terminal:
```bash
scp -P <port> -i ~/.ssh/id_ed25519_vast \
    'root@<vast-ip>:/root/torsion_scan_guardian/figures/sulf_phase5_vastai_*.png' \
    ./figures/
git add figures/sulf_phase5_vastai_*.png
git commit -m "Phase 5 demo on Vast.ai: figures"
git push
```

## 4. DESTROY the instance when done

From your **local** terminal:
```bash
vastai destroy instance <ID>
```
**`destroy` not `stop`** — destroying frees the disk too. Stopping keeps paying disk rent.

A forgotten RTX 4090 instance over a weekend = ~$15–20.